# Feature Selection Without Leakage

**Dataset:** `breast_cancer` | **Level:** advanced

Show how selecting features before cross validation inflates scores, and the correct nested version.

Sections: 1 Problem, 2 Method, 3 Setup and data, 4 Experiment, 5 Results, 6 Interpretation, 7 Limitations, 8 Exercises, 9 References.

## 1. Problem

Selecting features on all rows and then cross validating leaks label information into every fold. With many features and few rows the bias can be large.

## 2. Method

Compare (a) the wrong way: select the top k features by ANOVA F on the full data, then cross validate, and (b) the right way: select inside a pipeline fitted on each training fold. To make the effect visible we add 300 random noise columns to the real data and repeat 20 times. The noise columns are synthetic and are used only for this leakage demonstration.

## 3. Setup and data

Breast Cancer Wisconsin Diagnostic, 569 rows, 30 features, plus 300 synthetic noise columns for the leakage demonstration. The model comparison at the end uses only the real 30 features.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from ml_lab.data import load_dataset
from ml_lab.preprocessing import split_three_way, make_model_pipeline
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz
NB_ID = "feature_selection"
SEED = 42
set_seed(SEED); viz.apply_style(); print_environment()

python 3.13.18 | Linux-6.18.44-fc-v595-x86_64-with-glibc2.39
  numpy==2.5.3
  pandas==3.0.5
  scipy==1.18.1
  scikit-learn==1.9.1
  matplotlib==3.11.2
  xgboost==3.4.1
  lightgbm==4.7.0
  catboost==1.2.10
  umap-learn==0.6.0
  torch==2.15.0
  gymnasium==1.4.0


In [2]:
b = load_dataset("breast_cancer"); X, y = b.X, b.y
print(X.shape)

(569, 30)


## 4. Experiment

Sample 100 rows, add noise columns, and score with leaky and correct selection.

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
leaky, correct = [], []
for rep in range(20):
    rng = np.random.default_rng(rep)
    rows = rng.choice(len(X), 100, replace=False)
    Xs = np.hstack([X.values[rows], rng.normal(size=(100, 300))]); ys = y.values[rows]
    cv = StratifiedKFold(5, shuffle=True, random_state=rep)
    Xsel = SelectKBest(f_classif, k=10).fit_transform(Xs, ys)
    leaky.append(cross_val_score(LogisticRegression(max_iter=2000), StandardScaler().fit_transform(Xsel), ys, cv=cv).mean())
    correct.append(cross_val_score(make_pipeline(StandardScaler(), SelectKBest(f_classif, k=10), LogisticRegression(max_iter=2000)), Xs, ys, cv=cv).mean())
print("done")

done


## 5. Results

Compare accuracy of both procedures. Then, on pure noise labels, the leaky procedure should look better than chance and the correct one should not.

In [4]:
noise_leaky, noise_correct = [], []
for rep in range(20):
    rng = np.random.default_rng(100 + rep)
    Xn, yn = rng.normal(size=(100, 300)), rng.integers(0, 2, 100)
    cv = StratifiedKFold(5, shuffle=True, random_state=rep)
    noise_leaky.append(cross_val_score(LogisticRegression(max_iter=2000), StandardScaler().fit_transform(SelectKBest(f_classif, k=10).fit_transform(Xn, yn)), yn, cv=cv).mean())
    noise_correct.append(cross_val_score(make_pipeline(StandardScaler(), SelectKBest(f_classif, k=10), LogisticRegression(max_iter=2000)), Xn, yn, cv=cv).mean())
tab = pd.DataFrame({"real data + noise columns": [np.mean(leaky), np.mean(correct)], "pure noise labels (true accuracy about 0.5)": [np.mean(noise_leaky), np.mean(noise_correct)]}, index=["leaky selection", "selection inside pipeline"])
display(tab.round(3))
METRICS = {"real_leaky": float(np.mean(leaky)), "real_correct": float(np.mean(correct)), "noise_leaky": float(np.mean(noise_leaky)), "noise_correct": float(np.mean(noise_correct)), "noise_leaky_std": float(np.std(noise_leaky, ddof=1)), "noise_correct_std": float(np.std(noise_correct, ddof=1))}
CONFIG = {"k": 10, "noise_columns": 300, "rows": 100, "repetitions": 20}; EXTRA = {"table": tab.round(4).reset_index().to_dict("records")}; ALGORITHM = "SelectKBest in pipeline"

record = ExperimentRecord(notebook=NB_ID, family="advanced", algorithm=ALGORITHM, dataset="breast_cancer", task="classification", seed=SEED, config=CONFIG, metrics=METRICS, extra=EXTRA)
print("saved:", record.save())

,real data + noise columns,pure noise labels (true accuracy about 0.5)
leaky selection,0.936,0.740
selection inside pipeline,0.932,0.511


saved: experiments/results/feature_selection.json


## 6. Interpretation

On pure noise the true accuracy is 0.5. A leaky score above 0.5 is optimism created by the procedure, not signal. The in-pipeline score stays near 0.5 as it should.

## 7. Limitations

Synthetic noise is used by design to expose leakage and is not a real-world claim. Only one selector (univariate F) is tested.

## 8. Exercises

1. Replace SelectKBest with L1 logistic regression selection inside the pipeline.
2. Vary k and the number of noise columns and plot the leakage bias.
3. Do the same for scaling fitted on all rows.

## 9. References

Ambroise and McLachlan (2002), *Selection bias in gene extraction on the basis of microarray gene-expression data*, PNAS, is the standard reference for this effect. It is **not verified in the build environment**.